In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/mersad/protBuild")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from ml.src.training.dataset.protein_dataset import ProteinTextDataset, make_collate_fn
from ml.src.training.trainer.train import train_model_protein2text
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModel, AutoConfig
from ml.src.models.protein2Text.prot2text import Prot2Text
from torch.utils.data import DataLoader
from multiprocessing import Array
import pandas as pd
import random
import torch
import json
import math


In [3]:
pretrained_protein_model_name = 'facebook/esm2_t6_8M_UR50D'
pretrained_text_model_name = 'openai-community/gpt2'


In [4]:
protein_tokenizer = AutoTokenizer.from_pretrained(pretrained_protein_model_name, trust_remote_code=True)
text_tokenizer = AutoTokenizer.from_pretrained(pretrained_text_model_name, trust_remote_code=True)

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
protein_model = AutoModel.from_pretrained(pretrained_protein_model_name, trust_remote_code=True)

text_model_config = AutoConfig.from_pretrained(pretrained_text_model_name, add_cross_attention=True)
text_model = AutoModelForCausalLM.from_pretrained(pretrained_text_model_name, trust_remote_code=True, config=text_model_config)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                                                 | Status  | 
----------------------------------------------------+---------+-
transformer.h.{0...11}.crossattention.q_attn.bias   | MISSING | 
transformer.h.{0...11}.crossattention.c_attn.bias   | MISSING | 
transformer.h.{0...11}.crossattention.c_proj.bias   | MISSING | 
transformer.h.{0...11}.ln_cross_attn.weight         | MISSING | 
transformer.h.{0...11}.ln_cross_attn.bias           | MISSING | 
transformer.h.{0...11}.crossattention.c_proj.weight | MISSING | 
transformer.h.{0...11}.crossattention.q_attn.weight | MISSING | 
transformer.h.{0...11}.crossattention.c_attn.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
protein_model.config.max_position_embeddings


1026

In [7]:
text_model.config.max_position_embeddings

1024

In [8]:
protein_tokenizer.SPECIAL_TOKENS_ATTRIBUTES

['bos_token',
 'eos_token',
 'unk_token',
 'sep_token',
 'pad_token',
 'cls_token',
 'mask_token']

In [9]:
text_tokenizer.SPECIAL_TOKENS_ATTRIBUTES

['bos_token',
 'eos_token',
 'unk_token',
 'sep_token',
 'pad_token',
 'cls_token',
 'mask_token']

In [10]:
print(protein_tokenizer.pad_token_id)
print(protein_tokenizer.pad_token)

print(text_tokenizer.pad_token_id)
print(text_tokenizer.pad_token)

1
<pad>
None
None


In [11]:
protein_pad_id = protein_tokenizer.pad_token_id
if text_tokenizer.pad_token is None:
    text_tokenizer.pad_token = text_tokenizer.eos_token
text_pad_id = text_tokenizer.pad_token_id

In [12]:
num_workers=1
train_position_array = Array("Q", [0] * num_workers)
train_idx_array = Array("Q", [0] * num_workers)

In [13]:
train_dataset = ProteinTextDataset(
    PROJECT_ROOT / "ml/datasets/prot2text_train.jsonl",
    protein_tokenizer,
    text_tokenizer,
    1024,
    256,
    position_array=train_position_array,
    idx_array=train_idx_array,
)


val_dataset = ProteinTextDataset(
    PROJECT_ROOT / "ml/datasets/prot2text_val.jsonl",
    protein_tokenizer,
    text_tokenizer,
    1024,
    256,   
)

# test_dataset = ProteinTextDataset(
#     PROJECT_ROOT / "ml/datasets/prot2text_test.jsonl",
#     protein_tokenizer,
#     text_tokenizer,
#     1024,
#     1024,   
# )

In [14]:
collate_fn = make_collate_fn(
    protein_pad_id=protein_pad_id,
    text_pad_id=text_pad_id,
    ignore_index=-100,
)

In [15]:
val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    collate_fn=collate_fn,
    num_workers=1,
    pin_memory=True,
    persistent_workers=True,
)

# test_loader = DataLoader(
#     test_dataset,
#     batch_size=1,
#     collate_fn=collate_fn,
#     num_workers=1,
#     pin_memory=True,
#     persistent_workers=True,
# )

In [16]:
input_dim = protein_model.config.hidden_size
output_dim = text_model.config.hidden_size

In [17]:

model = Prot2Text(protein_model, text_model, 
                  input_dim=input_dim, hidden_dim=output_dim*2,
                    output_dim=output_dim, dropout=0.1,
                      bias=False)

In [18]:
for param in model.protein_model.parameters():
    param.requires_grad = False

trainable_params = [
    param for param in model.parameters()
    if param.requires_grad
]

print("Trainable parameters:",
      sum(p.numel() for p in trainable_params))

Trainable parameters: 154478592


In [19]:
optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=1e-4,
    weight_decay=0.01,
)

num_samples = 421_626
batch_size = 1
num_epochs = 5

steps_per_epoch = math.ceil(num_samples / batch_size)
total_steps = steps_per_epoch * num_epochs
warmup_steps = 20_000
min_lr_ratio = 1e-6 / 1e-4

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step + 1) / warmup_steps
    progress = (current_step - warmup_steps) / (total_steps - warmup_steps)

    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))

    return min_lr_ratio + (1.0 - min_lr_ratio) * cosine_decay

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lr_lambda
)


In [ ]:
metrics_history = train_model_protein2text(model=model, tokenizer=text_tokenizer, train_dataset=train_dataset,
                                            val_loader=val_loader, optimizer=optimizer, scheduler=scheduler, device=device,
                                            num_epochs=num_epochs, eval_freq=100000, eval_iter=2000, train_position_array=train_position_array, 
                                            train_idx_array=train_idx_array, checkpoint_iter=250, collate_fn=collate_fn, checkpoint_path='./checkpoints', 
                                            save_file_name='prot2text_checkpoint.pt', num_workers=num_workers, batch_size=batch_size)


Loading checkpoint from: ./checkpoints/prot2text_checkpoint.pt
Checkpoint loaded successfully.
Resuming from: 
Epoch: 2
Step: 492750
Worker resume state: 
{0: (54776550, 71126)}


Epoch 2/5: 0batch [00:00, ?batch/s][transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Epoch 2/5: 5batch [00:01,  4.59batch/s, loss=0.275][W1008 12:45:01.346444345 CUDACachingAllocator.cpp:3933] memory allocation failed with OOM on device 0 while trying to allocate 2097152 bytes (free: 5177344, total: 3953721344).
Epoch 2/5: 7249batch [19:02,  8.75batch/s, loss=0.006][transformers] The following layers were not sharded: encoder.layer.*.attention.self.query.weight, pooler.dense.weight, encoder.layer.*.attention.self.query.bias, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.output.LayerNorm.bias, embeddings.token_type_embeddings.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.attention.self.key.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.l

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 2 (Step 500000):
Train loss 0.535, Val loss 0.509
Train perplexity 1.707, Val perplexity 1.663
BLEU-4: 0.3964
ROUGE-1 F1: 0.6466
ROUGE-2 F1: 0.5814
ROUGE-L F1: 0.6380
BERTScore Precision: 0.6294
BERTScore Recall: 0.5859
BERTScore F1: 0.6007


Epoch 2/5: 107249batch [5:04:33,  7.10batch/s, loss=2.234][transformers] The following layers were not sharded: encoder.layer.*.attention.self.query.weight, pooler.dense.weight, encoder.layer.*.attention.self.query.bias, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.output.LayerNorm.bias, embeddings.token_type_embeddings.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.attention.self.key.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.self.value.weight, encoder.layer.*.attention.output.dense.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.self.value.bias, encoder.layer.*.attention.self.key.bias, pooler.dense.bias, embeddings.LayerNorm.weight, encoder.layer.*.output.dense.bias, embeddings.LayerNorm.bias, embeddings.position_embeddin

...

Epoch 3 (Step 900000):
Train loss 0.428, Val loss 0.416
Train perplexity 1.534, Val perplexity 1.516
BLEU-4: 0.4658
ROUGE-1 F1: 0.6904
ROUGE-2 F1: 0.6356
ROUGE-L F1: 0.6823
BERTScore Precision: 0.6244
BERTScore Recall: 0.5875
BERTScore F1: 0.5990


Epoch 3/5: 156749batch [7:29:53,  6.99batch/s, loss=0.012][transformers] The following layers were not sharded: encoder.layer.*.attention.self.query.weight, pooler.dense.weight, encoder.layer.*.attention.self.query.bias, encoder.layer.*.output.LayerNorm.bias, encoder.layer.*.attention.output.dense.weight, encoder.layer.*.output.dense.weight, encoder.layer.*.attention.output.LayerNorm.bias, embeddings.token_type_embeddings.weight, encoder.layer.*.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.bias, encoder.layer.*.attention.self.key.weight, encoder.layer.*.attention.output.LayerNorm.weight, encoder.layer.*.intermediate.dense.weight, encoder.layer.*.attention.self.value.weight, encoder.layer.*.attention.output.dense.bias, embeddings.word_embeddings.weight, encoder.layer.*.attention.self.value.bias, encoder.layer.*.attention.self.key.bias, pooler.dense.bias, embeddings.LayerNorm.weight, encoder.layer.*.output.dense.bias, embeddings.LayerNorm.bias, embeddings.position_embeddin

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Epoch 3 (Step 1000000):
Train loss 0.405, Val loss 0.397
Train perplexity 1.499, Val perplexity 1.487
BLEU-4: 0.3013
ROUGE-1 F1: 0.6179
ROUGE-2 F1: 0.5706
ROUGE-L F1: 0.6130
BERTScore Precision: 0.6307
BERTScore Recall: 0.5559
BERTScore F1: 0.5843


Epoch 3/5: 245249batch [11:49:23,  8.60batch/s, loss=0.115]

In [ ]:
torch.save(model.state_dict(), PROJECT_ROOT/'ml/notebooks/prot2text_model.pth')

In [ ]:
df = pd.DataFrame.from_dict(metrics_history, orient="index")
df.index.name = "step"
df.to_csv(PROJECT_ROOT/'ml/notebooks/training_history.csv')